# **Scenario 1: Design a Distributed Job Scheduling and Execution Engine**

**The Context:**
You are building an internal "Task Execution Platform" for a fast-growing e-commerce company. Developers from various teams will submit background jobs to this system—anything from generating daily financial reports and processing large image batches, to sending out millions of promotional emails. 

**Initial Parameters:**
* **Scale:** The system receives roughly 10,000 job submissions per minute. 
* **Workload Variance:** Job execution times are highly variable. Some take 5 seconds; others take 2 hours. 
* **Isolation:** Since different teams write these Python jobs, they might have conflicting dependencies. A crash in one job must never take down the system or affect other running jobs.
* **Reliability:** Jobs cannot be lost if a server crashes. The system needs to guarantee at least *at-least-once* execution, with a preference for handling retries gracefully.
* **Operations (The DevOps Angle):** You need a strategy for auto-scaling the underlying compute resources based on queue depth, and a safe way to deploy updates to the worker nodes without killing currently processing, long-running jobs.

### 1. The Core Context: What is the System's Purpose?

**The Business Need:**
Imagine a large e-commerce company. The main website (the part customers see) needs to be fast and responsive. However, there are many heavy tasks that need to happen that would slow down the website if done immediately.
*   **Example:** When a user uploads a profile picture, the website shouldn't freeze while resizing that image. It should say "Upload Complete" and then handle the resizing in the background.
*   **Example:** At the end of the day, the finance team needs a report summarizing millions of transactions. This takes a long time and shouldn't block the website.

**Your System's Role:**
You are building the "background worker" system. Developers from other teams will send tasks to your system, and your system is responsible for finding a computer to run that task, running it, and reporting back when it's done. You are essentially building a **factory for code execution**.

**The Multi-Tenant Aspect:**
Many different teams (Finance, Marketing, Engineering) will use your system. They do not coordinate with each other. One team might write efficient code; another might write code that leaks memory. Your system must treat all of them fairly and safely.

---

### 2. Scale: The Ingestion Pressure

**Requirement:** `10,000 job submissions per minute`

**What this challenges you to solve:**
*   **The Entry Point:** Your system needs an "entry point" (an API) where developers send their jobs. This entry point must be able to accept 10,000 requests every minute without slowing down or rejecting them.
*   **Buffering:** Since jobs take time to process (seconds to hours), you cannot process them instantly as they arrive. You need a place to store these jobs temporarily while they wait for a worker to pick them up. This storage needs to be fast enough to handle the write speed of 10,000 items per minute.
*   **Backpressure:** What happens if jobs are coming in faster than they can be processed? The waiting line (queue) will grow. Your system needs to handle a growing line without crashing.

**Key Question for You:** How do you ensure the system accepts jobs quickly even if the processing side is slow?

---

### 3. Workload Variance: The Time Problem

**Requirement:** `Job execution times are highly variable. Some take 5 seconds; others take 2 hours.`

**What this challenges you to solve:**
*   **Resource Hoarding:** A job that takes 2 hours occupies a worker machine for a long time. If you have many 2-hour jobs running, you might not have enough workers left to handle the 5-second jobs.
*   **Efficiency:** Is it efficient to use the same type of worker machine for a 5-second task as you do for a 2-hour task? Maybe not.
*   **Visibility:** If a job runs for 2 hours, how do you know it's still alive? What if it got stuck after 1 hour? Your system needs a way to distinguish between "still working" and "dead/stuck."
*   **Queue Blocking:** If you have a single line for all jobs, a long job at the front of the line might delay thousands of short jobs behind it.

**Key Question for You:** How do you manage workers so that short jobs aren't delayed by long jobs, and how do you track the status of tasks that run for hours?

---

### 4. Isolation: The Safety Problem

**Requirement:** `Conflicting dependencies. A crash in one job must never take down the system or affect other running jobs.`

**What this challenges you to solve:**
*   **Library Conflicts:** Team A's job needs Version 1 of a software library. Team B's job needs Version 2. They cannot exist on the same standard environment simultaneously.
*   **Resource Contention:** If one job goes into an infinite loop and uses 100% of the CPU, it shouldn't slow down the other jobs running on the same physical machine.
*   **Memory Safety:** If one job leaks memory and fills up the RAM, it shouldn't cause the other jobs to crash due to lack of memory.
*   **Security:** One team's job should not be able to read the data or environment variables of another team's job.

**Key Question for You:** How do you run untrusted code from different teams on the same physical hardware without them interfering with each other?

---

### 5. Reliability: The Failure Problem

**Requirement:** `Jobs cannot be lost if a server crashes. At-least-once execution. Handle retries gracefully.`

**What this challenges you to solve:**
*   **Durability:** When a job is submitted, it must be saved permanently. If your entire system loses power immediately after submission, that job must still exist when the system turns back on.
*   **Worker Failure:** Workers (the machines running the code) will crash. It is a fact of distributed systems. If a worker crashes while running a job, that job needs to be picked up by a different worker.
*   **Double Execution:** Because workers crash, you might end up running a job twice (e.g., Worker A starts, crashes, Worker B picks it up). Your system must allow for this possibility. This means the code running inside the job needs to be safe to run multiple times (idempotent), or your system needs to prevent double execution.
*   **Retry Logic:** If a job fails because of a temporary glitch (like a network timeout), you should try again. If it fails because of a bug in the code, retrying won't help. You need a way to distinguish these and stop retrying eventually.

**Key Question for You:** How do you ensure a job is never lost, even if the machine running it dies, while acknowledging that a job might run more than once?

---

### 6. Operations: The Maintenance Problem

**Requirement:** `Auto-scaling based on queue depth. Safe way to deploy updates without killing long-running jobs.`

**What this challenges you to solve:**
*   **Cost vs. Performance:** You don't want to pay for 1,000 workers when there are no jobs. You don't want 10 workers when there are 1 million jobs. You need a mechanism to add/remove workers automatically based on how much work is waiting.
*   **Updating the System:** You will need to update the software that runs the workers.
    *   **The Risk:** If you restart all workers to update the software, you will kill any job that is currently running (especially those 2-hour jobs).
    *   **The Challenge:** You need a way to update the software on a worker only after it has finished its current task, not while it is in the middle of one.

**Key Question for You:** How do you add/remove capacity automatically, and how do you update the worker software without interrupting tasks that are already in progress?

---

### Summary of Tensions (Trade-offs to Consider)

As you design this, you will find that some requirements fight against each other. Here are the main tensions you need to balance:

1.  **Speed vs. Safety:** To make things fast, you might want to run many jobs on one machine. To make things safe (Isolation), you want to separate them, which uses more resources.
2.  **Efficiency vs. Reliability:** To be efficient, you want to run a job exactly once. To be reliable (in case of crashes), you must be prepared to run it at least once, which risks running it twice.
3.  **Utilization vs. Latency:** To save money, you want workers to be 100% busy. But if they are 100% busy, new jobs have to wait in line (increasing latency).
4.  **Deployment Speed vs. Job Continuity:** To deploy updates fast, you want to restart workers immediately. To protect long jobs, you need to wait for them to finish before restarting.

### Final Thought for Your Design

When you start drawing your diagram or writing your solution, keep this narrative in mind:
*"I need to accept a high volume of tasks, store them safely, distribute them to isolated environments that can run different types of code, ensure they finish even if machines break, and manage the fleet of machines efficiently without interrupting work."*